In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import bacco
from matplotlib.colors import LogNorm
from scipy.ndimage import gaussian_filter

In [ ]:
plt.rcParams["font.family"] = "serif"
plt.rcParams["mathtext.fontset"] = "dejavuserif"

In [ ]:
## Load the Zooms
sigma8 = 0.8159 #CHECK ME
ns     = 0.9667 #CHECK ME
tau    = 0.0965 #CHECK ME

#name_list = ['LH_{:d}'.format(i) for i in range(30)] + ['fiducial'] + ['bf_sim']
name_list = ['bf_sim']

snap = 264

name_list = ['fiducial']

zoom = {}
for i in range(len(name_list)):
    base = "/cosmos_storage/simulations/TNG_Family/MN5_resims/"+name_list[i]+"/hydro_output/"
    zoom[name_list[i]] = bacco.Simulation(basedir=base, halo_file="groups_{:03d}/fof_subhalo_tab_{:03d}".format(snap,snap), sim_format='TNG500', fixedPk=True, use_orphans=False,\
                            tau=tau, ns=ns, sigma8=sigma8, dm_file="snapdir_{:03d}/snapshot_{:03d}".format(snap,snap), use_ids=True, numpart=4320)



In [ ]:
dm_pos = zoom['fiducial'].dm['pos']
dm_mass = 1e10 * np.ones(dm_pos.shape[0]) * zoom['fiducial'].header['ParticleMass']
mask = (dm_pos[:,2] > 200) & (dm_pos[:,2] < 450)

lowres_pos = zoom['fiducial'].lowres_dm['pos']
lowres_mass = 1e10 * zoom['fiducial'].lowres_dm['mass']
mask_lowres = (lowres_pos[:,2] > 200) & (lowres_pos[:,2] < 450) 

xy_range = [[0, 500], [0, 500]]

cell = 500 / 1000

hist_dm = np.histogram2d(dm_pos[mask][:,0], dm_pos[mask][:,1], weights=dm_mass[mask], bins=1000, range=xy_range)
hist_lowres = np.histogram2d(lowres_pos[mask_lowres][:,0],\
    lowres_pos[mask_lowres][:,1],\
    weights=lowres_mass[mask_lowres], bins=1000, range=xy_range)

In [ ]:
zoom['fiducial'].header['OmegaBaryon'] / zoom['fiducial'].header['Omega']

In [ ]:
fig, ax = plt.subplots(dpi=200, figsize=(6, 5))

# 1. Transpose the histograms to match imshow's (y, x) expectation
h_dm = hist_dm[0].T / cell**3
h_lowres = hist_lowres[0].T / cell**3

# 2. Smooth the low-res data to reduce shot noise
h_lowres_smooth = gaussian_filter(h_lowres, sigma=4)

# 3. Define the physical extent of your box for the axes labels
box_size = 500
extent = [0, box_size, 0, box_size]

# 4. Plot High-Res DM
vmin_dm = np.median(dm_mass) # A good starting guess for vmin
im_dm = ax.imshow(h_dm, cmap='inferno', 
                  norm=LogNorm(vmin=vmin_dm, vmax=np.max(h_dm)), 
                  origin='lower', extent=extent)

# Mask out empty regions in the smoothed low-res map so it doesn't block the high-res
h_lowres_masked = np.ma.masked_where(h_dm > 0, h_lowres_smooth)

im_lowres = ax.imshow(h_lowres_masked, cmap='binary', 
                      norm=LogNorm(vmin=10**13, vmax=10**14.2),
                      origin='lower', extent=extent, alpha=1)

# 6. Formatting
# ax.set_facecolor('black') # Ensures absolute empty space is black, not white
ax.set_xlabel("x [Mpc/h]")
ax.set_ylabel("y [Mpc/h]")
ax.set_title("DM Density Projection ($\Delta z = 250$ [Mpc/$h$])")

# Add a colorbar for the high-res field
cbar = fig.colorbar(im_dm, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('High-Res DM Density [$M_\odot h^{-1} / ($Mpc$/h)^3$]')

plt.tight_layout()
plt.show()

In [ ]:
zoom['fiducial'].fof['halo_pos'][0]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.ndimage import gaussian_filter

# ... [Your data loading and histogram generation remains the same] ...
# h_dm = hist_dm[0].T
# h_lowres = hist_lowres[0].T
# extent = [0, box_size, 0, box_size]

fig, ax = plt.subplots(dpi=200, figsize=(8, 8))

# 1. Base Layer: High-Res DM (Same as before)
vmin_dm = np.median(dm_mass) 
im_dm = ax.imshow(h_dm, cmap='inferno', 
                  norm=LogNorm(vmin=vmin_dm, vmax=np.max(h_dm)), 
                  origin='lower', extent=extent)

# 2. Smooth the Low-Res DM 
# For contours, a slightly higher sigma (e.g., 3.0 to 5.0) often looks better 
# so the lines trace the broad filaments rather than jagged individual halos.
h_lowres_smooth = gaussian_filter(h_lowres, sigma=3.0)

# 3. Define Logarithmic Contour Levels
# Cosmological density drops off exponentially, so linear contours won't work.
# We generate e.g., 5 or 6 levels spaced logarithmically.
vmin_contour = np.median(lowres_mass) * 2  # Start slightly above the noise floor
vmax_contour = np.max(h_lowres_smooth) / 2 # Stop slightly before the absolute densest peak
levels = np.logspace(np.log10(vmin_contour), np.log10(vmax_contour), num=6)

# 4. Plot the Contours
# We use a solid color (like white, cyan, or light gray) so it pops against 'inferno'
contours = ax.contour(h_lowres_smooth, 
                      levels=levels, 
                      colors='white', 
                      alpha=0.6, 
                      linewidths=0.8,
                      extent=extent) # Extent ensures it aligns perfectly with imshow

# 5. Formatting
ax.set_facecolor('black')
ax.set_xlabel("x [cMpc/h]")
ax.set_ylabel("y [cMpc/h]")
ax.set_title("DM Density: High-Res Core with Low-Res Structure Contours")

# Base layer colorbar
cbar = fig.colorbar(im_dm, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('High-res Mass Density')

plt.tight_layout()
plt.show()